# ChurnIQ — Telecom Customer Churn & Revenue Risk Analyzer

## Phase 6.1 — Modeling Dataset Preparation

### Objective

Prepare the feature-engineered dataset for machine learning by:

- Defining the target variable
- Selecting modeling features
- Creating Model A and Model B datasets
- Designing train-validation strategy
- Validating class distributions

---

### Business Goal

Predict customer churn and support retention decision-making through data-driven prioritization.

---

### Modeling Strategy

#### Model A
Includes retention-related variables to evaluate maximum predictive performance.

#### Model B
Excludes retention-related variables to evaluate a deployment-safe model with reduced leakage risk.

---

### Dataset

Source: `data/processed/feature_engineered.csv`

Target Variable: `Churn`

Current Dataset Status:

- Data cleaned
- Missing values handled
- Feature engineered
- Ready for modeling

In [4]:
# ==========================================
# IMPORT LIBRARIES
# ==========================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

# Display settings

pd.set_option("display.max_columns", None)

print("Libraries loaded successfully.")

Libraries loaded successfully.


## Step 1: Load Feature Engineered Dataset

Load the final dataset produced after Phase 5.

This dataset already contains:

- Cleaned data
- Imputed values
- Engineered features
- Validated business features

The purpose of this step is to verify that the dataset is ready for modeling.

In [5]:
# ==========================================
# LOAD DATASET
# ==========================================

df = pd.read_csv("../data/processed/feature_engineered.csv")

print("Dataset Loaded Successfully")
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

Dataset Loaded Successfully
Rows    : 51047
Columns : 67


C:\Users\Akshit\AppData\Local\Temp\ipykernel_1368\496843601.py:5: DtypeWarning: Columns (66) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/feature_engineered.csv")


## Step 2: Initial Dataset Validation

Before preparing the modeling dataset, verify:

1. Dataset shape
2. Target variable availability
3. Target distribution
4. Remaining missing values

This ensures that the dataset entering the modeling phase is trustworthy.

In [6]:
# Dataset Shape

print("Dataset Shape:")
print(df.shape)

Dataset Shape:
(51047, 67)


In [7]:
# Target Distribution

print("Target Distribution:")
print(df["Churn"].value_counts())

Target Distribution:
Churn
No     36336
Yes    14711
Name: count, dtype: int64


In [8]:
# Target Distribution (%)

print("Target Distribution (%):")

round(
    df["Churn"]
    .value_counts(normalize=True)
    .mul(100),
    2
)

Target Distribution (%):


Churn
No     71.18
Yes    28.82
Name: proportion, dtype: float64

In [9]:
# Missing Values Check

print("Total Missing Values Remaining:")
print(df.isnull().sum().sum())

Total Missing Values Remaining:
0


In [10]:
# Preview Dataset

df.head()

,CustomerID,Churn,MonthlyRevenue,MonthlyMinutes,TotalRecurringCharge,DirectorAssistedCalls,OverageMinutes,RoamingCalls,PercChangeMinutes,PercChangeRevenues,DroppedCalls,BlockedCalls,UnansweredCalls,CustomerCareCalls,ThreewayCalls,ReceivedCalls,OutboundCalls,InboundCalls,PeakCallsInOut,OffPeakCallsInOut,DroppedBlockedCalls,CallForwardingCalls,CallWaitingCalls,MonthsInService,UniqueSubs,ActiveSubs,ServiceArea,Handsets,HandsetModels,CurrentEquipmentDays,AgeHH1,AgeHH2,ChildrenInHH,HandsetRefurbished,HandsetWebCapable,TruckOwner,RVOwner,Homeownership,BuysViaMailOrder,RespondsToMailOffers,OptOutMailings,NonUSTravel,OwnsComputer,HasCreditCard,RetentionCalls,RetentionOffersAccepted,NewCellphoneUser,NotNewCellphoneUser,ReferralsMadeBySubscriber,IncomeGroup,OwnsMotorcycle,AdjustmentsToCreditRating,HandsetPrice,MadeCallToRetentionTeam,CreditRating,PrizmCode,Occupation,MaritalStatus,AgeHH2_Missing,HandsetPrice_Missing,RevenueGroup_Missing,CustomerLifecycleStage,CustomerValueProxy,CreditRating_Encoded,RegionCode,MarketCode,AreaCode
0,3000002,Yes,24.00,219.0,22.0,0.25,0.0,0.0,-157.0,-19.0,0.7,0.7,6.3,0.0,0.0,97.2,0.0,0.0,58.0,24.0,1.3,0.0,0.3,61,2,1,SEAPOR503,2.0,2.0,361.0,62.0,44.0,No,No,Yes,No,No,Known,Yes,Yes,No,No,Yes,Yes,1,0,No,No,0,4,No,0,30.0,Yes,1-Highest,Suburban,Professional,No,True,0,0,Loyal,1464.00,7,SEA,POR,503
1,3000010,Yes,16.99,10.0,17.0,0.00,0.0,0.0,-4.0,0.0,0.3,0.0,2.7,0.0,0.0,0.0,0.0,0.0,5.0,1.0,0.3,0.0,0.0,58,1,1,PITHOM412,2.0,1.0,1504.0,40.0,42.0,Yes,No,No,No,No,Known,Yes,Yes,No,No,Yes,Yes,0,0,Yes,No,0,5,No,0,30.0,No,4-Medium,Suburban,Professional,Yes,False,0,0,Loyal,985.42,4,PIT,HOM,412
2,3000014,No,38.00,8.0,38.0,0.00,0.0,0.0,-2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.4,0.3,0.0,1.3,3.7,0.0,0.0,0.0,60,1,1,MILMIL414,1.0,1.0,1812.0,26.0,26.0,Yes,No,No,No,No,Unknown,No,No,No,No,No,Yes,0,0,Yes,No,0,6,No,0,60.0,No,3-Good,Town,Crafts,Yes,False,1,0,Loyal,2280.00,5,MIL,MIL,414
3,3000022,No,82.28,1312.0,75.0,1.24,0.0,0.0,157.0,8.1,52.0,7.7,76.0,4.3,1.3,200.3,370.3,147.0,555.7,303.7,59.7,0.0,22.7,59,2,2,PITHOM412,9.0,4.0,458.0,30.0,44.0,No,No,Yes,No,No,Known,Yes,Yes,No,No,No,Yes,0,0,Yes,No,0,6,No,0,10.0,No,4-Medium,Other,Other,No,True,0,0,Loyal,4854.52,4,PIT,HOM,412
4,3000026,Yes,17.14,0.0,17.0,0.00,0.0,0.0,0.0,-0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,53,2,2,OKCTUL918,4.0,3.0,852.0,46.0,54.0,No,No,No,No,No,Known,Yes,Yes,No,No,Yes,Yes,0,0,No,Yes,0,9,No,1,10.0,No,1-Highest,Other,Professional,Yes,False,0,0,Loyal,908.42,7,OKC,TUL,918


## Step 3: Target Variable Preparation

Machine learning algorithms require a numerical target variable.

The original target column:

- Yes
- No

will be converted into:

- Yes → 1
- No → 0

This preserves the business meaning while enabling model training and probability estimation.

In [12]:
# ==========================================
# TARGET ENCODING
# ==========================================

df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

print("Target encoded successfully.\n")

print(df["Churn"].value_counts())

Target encoded successfully.

Churn
0    36336
1    14711
Name: count, dtype: int64


In [13]:
# Verify encoding

df["Churn"].head()

0    1
1    1
2    0
3    0
4    1
Name: Churn, dtype: int64

## Step 4: Feature Selection Strategy

Not every column should be used for modeling.

### Columns Removed

#### CustomerID

CustomerID is a unique identifier.

It does not represent customer behavior and therefore provides no meaningful predictive signal.

#### ServiceArea

The original ServiceArea field was decomposed into:

- RegionCode
- MarketCode
- AreaCode

Keeping both the original column and derived columns would introduce redundancy.

Therefore, ServiceArea will be removed from modeling datasets.

In [15]:
# ==========================================
# INITIAL COLUMN EXCLUSIONS
# ==========================================

columns_to_drop = [
    "CustomerID",
    "ServiceArea"
]

print("Columns scheduled for removal:")
print(columns_to_drop)

Columns scheduled for removal:
['CustomerID', 'ServiceArea']


In [16]:
# Create base modeling dataframe

model_df = df.drop(columns=columns_to_drop)

print("Modeling Dataset Shape:")
print(model_df.shape)

Modeling Dataset Shape:
(51047, 65)


In [17]:
# Verify removed columns

removed_columns = [
    col for col in columns_to_drop
    if col not in model_df.columns
]

print("Successfully Removed:")
print(removed_columns)

Successfully Removed:
['CustomerID', 'ServiceArea']


## Step 5: Feature Inventory Audit

Before creating modeling datasets, perform a feature inventory.

Objectives:

- Identify numerical features
- Identify categorical features
- Detect high-cardinality variables
- Understand encoding requirements

This step helps design a robust modeling pipeline and prevents surprises during model training.

In [18]:
# ==========================================
# FEATURE TYPE AUDIT
# ==========================================

categorical_cols = model_df.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_cols = model_df.select_dtypes(
    exclude=["object"]
).columns.tolist()

print(f"Categorical Features : {len(categorical_cols)}")
print(f"Numerical Features   : {len(numerical_cols)}")

Categorical Features : 24
Numerical Features   : 41


In [19]:
# Categorical Columns

categorical_cols

['ChildrenInHH',
 'HandsetRefurbished',
 'HandsetWebCapable',
 'TruckOwner',
 'RVOwner',
 'Homeownership',
 'BuysViaMailOrder',
 'RespondsToMailOffers',
 'OptOutMailings',
 'NonUSTravel',
 'OwnsComputer',
 'HasCreditCard',
 'NewCellphoneUser',
 'NotNewCellphoneUser',
 'OwnsMotorcycle',
 'MadeCallToRetentionTeam',
 'CreditRating',
 'PrizmCode',
 'Occupation',
 'MaritalStatus',
 'CustomerLifecycleStage',
 'RegionCode',
 'MarketCode',
 'AreaCode']

In [20]:
# Numerical Columns

numerical_cols[:20]

['Churn',
 'MonthlyRevenue',
 'MonthlyMinutes',
 'TotalRecurringCharge',
 'DirectorAssistedCalls',
 'OverageMinutes',
 'RoamingCalls',
 'PercChangeMinutes',
 'PercChangeRevenues',
 'DroppedCalls',
 'BlockedCalls',
 'UnansweredCalls',
 'CustomerCareCalls',
 'ThreewayCalls',
 'ReceivedCalls',
 'OutboundCalls',
 'InboundCalls',
 'PeakCallsInOut',
 'OffPeakCallsInOut',
 'DroppedBlockedCalls']

In [23]:
# ==========================================
# HIGH CARDINALITY CHECK
# ==========================================

cardinality_df = pd.DataFrame({
    "Feature": categorical_cols,
    "Unique_Values": [
        model_df[col].nunique()
        for col in categorical_cols
    ]
})

cardinality_df = cardinality_df.sort_values(
    by="Unique_Values",
    ascending=False
)

cardinality_df

,Feature,Unique_Values
22,MarketCode,534
23,AreaCode,360
21,RegionCode,58
18,Occupation,8
16,CreditRating,7
17,PrizmCode,4
20,CustomerLifecycleStage,3
19,MaritalStatus,3
1,HandsetRefurbished,2
15,MadeCallToRetentionTeam,2


In [24]:
# Top 10 highest-cardinality categorical features

cardinality_df.head(10)

,Feature,Unique_Values
22,MarketCode,534
23,AreaCode,360
21,RegionCode,58
18,Occupation,8
16,CreditRating,7
17,PrizmCode,4
20,CustomerLifecycleStage,3
19,MaritalStatus,3
1,HandsetRefurbished,2
15,MadeCallToRetentionTeam,2


## Step 6: High Cardinality Feature Handling

Feature inventory revealed extremely high-cardinality categorical variables.

### Removed

- MarketCode (534 categories)
- AreaCode (360 categories)

Reason:

These variables would significantly increase dimensionality after encoding and may lead to overfitting.

### Retained

- RegionCode
- Occupation
- CreditRating
- PrizmCode
- CustomerLifecycleStage
- MaritalStatus

These variables provide business-relevant information while maintaining manageable cardinality.

In [26]:
# ==========================================
# HIGH CARDINALITY FEATURE REMOVAL
# ==========================================

high_cardinality_cols = [
    "MarketCode",
    "AreaCode"
]

model_df = model_df.drop(
    columns=high_cardinality_cols
)

print("Removed High Cardinality Features:")
print(high_cardinality_cols)

print("\nUpdated Shape:")
print(model_df.shape)

Removed High Cardinality Features:
['MarketCode', 'AreaCode']

Updated Shape:
(51047, 63)


In [27]:
# Verify removal

for col in high_cardinality_cols:
    print(f"{col} present:",
          col in model_df.columns)

MarketCode present: False
AreaCode present: False


## Step 7: Retention Feature Strategy

Certain variables may contain information generated after a customer becomes a churn risk.

These variables are:

- RetentionCalls
- RetentionOffersAccepted
- MadeCallToRetentionTeam

Their predictive power may be artificially inflated because retention actions often occur after churn risk is identified.

To evaluate this possibility, two modeling datasets will be created.

### Model A

Includes retention-related variables.

Purpose:
Measure maximum predictive performance.

### Model B

Excludes retention-related variables.

Purpose:
Measure performance without potential leakage-related features.

The two models will later be compared to determine whether retention variables introduce unrealistic predictive gains.m

In [29]:
# ==========================================
# RETENTION FEATURES
# ==========================================

retention_features = [
    "RetentionCalls",
    "RetentionOffersAccepted",
    "MadeCallToRetentionTeam"
]

print("Retention Features:")
print(retention_features)

Retention Features:
['RetentionCalls', 'RetentionOffersAccepted', 'MadeCallToRetentionTeam']


In [30]:
# Verify availability

for col in retention_features:
    print(f"{col}:",
          col in model_df.columns)

RetentionCalls: True
RetentionOffersAccepted: True
MadeCallToRetentionTeam: True


In [31]:
# ==========================================
# MODEL A DATASET
# ==========================================

model_a_df = model_df.copy()

print("Model A Shape:")
print(model_a_df.shape)

Model A Shape:
(51047, 63)


In [32]:
# ==========================================
# MODEL B DATASET
# ==========================================

model_b_df = model_df.drop(
    columns=retention_features
)

print("Model B Shape:")
print(model_b_df.shape)


Model B Shape:
(51047, 60)


In [33]:
print("Model A Columns:", model_a_df.shape[1])
print("Model B Columns:", model_b_df.shape[1])

Model A Columns: 63
Model B Columns: 60


## Step 8: Define Features and Target

Machine learning models require a clear separation between:

- Features (X)
- Target (y)

Target Variable:

- Churn

Feature Variables:

- All remaining predictor variables

Separate feature matrices and target vectors will be created for both Model A and Model B.

In [34]:
# ==========================================
# MODEL A : FEATURES & TARGET
# ==========================================

X_a = model_a_df.drop(columns=["Churn"])
y_a = model_a_df["Churn"]

print("X_a Shape:", X_a.shape)
print("y_a Shape:", y_a.shape)

X_a Shape: (51047, 62)
y_a Shape: (51047,)


In [35]:
# ==========================================
# MODEL B : FEATURES & TARGET
# ==========================================

X_b = model_b_df.drop(columns=["Churn"])
y_b = model_b_df["Churn"]

print("X_b Shape:", X_b.shape)
print("y_b Shape:", y_b.shape)

X_b Shape: (51047, 59)
y_b Shape: (51047,)


In [36]:
# Verify target distribution

print("Model A Target Distribution (%)")

print(
    round(
        y_a.value_counts(normalize=True) * 100,
        2
    )
)

Model A Target Distribution (%)
Churn
0    71.18
1    28.82
Name: proportion, dtype: float64


In [37]:
print("Model B Target Distribution (%)")

print(
    round(
        y_b.value_counts(normalize=True) * 100,
        2
    )
)

Model B Target Distribution (%)
Churn
0    71.18
1    28.82
Name: proportion, dtype: float64


## Step 9: Train-Validation Split

The dataset will be divided into:

- Training Set (80%)
- Validation Set (20%)

A stratified split is used to preserve the original churn distribution in both datasets.

Benefits:

- Prevents class imbalance distortion
- Produces realistic model evaluation
- Follows industry-standard machine learning practice

In [38]:
# ==========================================
# MODEL A TRAIN / VALIDATION SPLIT
# ==========================================

X_train_a, X_valid_a, y_train_a, y_valid_a = train_test_split(
    X_a,
    y_a,
    test_size=0.20,
    stratify=y_a,
    random_state=42
)

print("Model A Split Complete")

Model A Split Complete


In [39]:
print("X_train_a :", X_train_a.shape)
print("X_valid_a :", X_valid_a.shape)

print("y_train_a :", y_train_a.shape)
print("y_valid_a :", y_valid_a.shape)

X_train_a : (40837, 62)
X_valid_a : (10210, 62)
y_train_a : (40837,)
y_valid_a : (10210,)


In [40]:
# ==========================================
# MODEL B TRAIN / VALIDATION SPLIT
# ==========================================

X_train_b, X_valid_b, y_train_b, y_valid_b = train_test_split(
    X_b,
    y_b,
    test_size=0.20,
    stratify=y_b,
    random_state=42
)

print("Model B Split Complete")

Model B Split Complete


In [41]:
print("X_train_b :", X_train_b.shape)
print("X_valid_b :", X_valid_b.shape)

print("y_train_b :", y_train_b.shape)
print("y_valid_b :", y_valid_b.shape)

X_train_b : (40837, 59)
X_valid_b : (10210, 59)
y_train_b : (40837,)
y_valid_b : (10210,)


## Step 10: Split Validation

After creating train and validation datasets, verify that the churn distribution remains consistent.

This confirms that stratification worked correctly.

In [42]:
print("Train Churn Distribution (%)")

round(
    y_train_a.value_counts(normalize=True) * 100,
    2
)

Train Churn Distribution (%)


Churn
0    71.18
1    28.82
Name: proportion, dtype: float64

In [43]:
print("Validation Churn Distribution (%)")

round(
    y_valid_a.value_counts(normalize=True) * 100,
    2
)

Validation Churn Distribution (%)


Churn
0    71.19
1    28.81
Name: proportion, dtype: float64